# Actividad 5: Método de uniformización

Sea la matriz de tasas:

$$
R=\begin{pmatrix}
0&2&3&0\\
4&0&2&0\\
0&2&0&2\\
1&0&3&0
\end{pmatrix}.
$$

Para el método de uniformización se usa

$$
r=\max_i r_i,
\qquad r_i=\sum_{j=1}^N r_{i,j},
$$

y

$$
\widehat p_{i,j}=\begin{cases}
1-\dfrac{r_i}{r},& i=j,\\[6pt]
\dfrac{r_{i,j}}{r},& i\neq j.
\end{cases}
$$

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=8, suppress=True)

R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

tasas = R.sum(axis=1)
r = tasas.max()

P_hat = R / r
np.fill_diagonal(P_hat, 1 - tasas / r)

print('tasas =', tasas)
print('r =', r)
print('\nP_hat =')
print(P_hat)
print('\nSuma por renglón de P_hat =', P_hat.sum(axis=1))

tasas = [5. 6. 4. 4.]
r = 6.0

P_hat =
[[0.16666667 0.33333333 0.5        0.        ]
 [0.66666667 0.         0.33333333 0.        ]
 [0.         0.33333333 0.33333333 0.33333333]
 [0.16666667 0.         0.5        0.33333333]]

Suma por renglón de P_hat = [1. 1. 1. 1.]


## Ejercicio 3

Se debe calcular $P(0.5)$, $P(1)$ y $P(5)$ usando la aproximación

$$
P(t)\approx P^M(t)=\sum_{k=0}^{M}e^{-rt}\frac{(rt)^k}{k!}\widehat P^k.
$$

Donde, 

$$
M\approx \max\{rt+5\sqrt{rt},20\}.
$$

Tomemos:

$$
M=\left\lceil\max\{rt+5\sqrt{rt},20\}\right\rceil.
$$

In [2]:
def M_propuesto(r, t):
    rt = r * t
    return int(np.ceil(max(rt + 5*np.sqrt(rt), 20)))

def uniformizacion_M(P_hat, r, t, M):
    n = P_hat.shape[0]
    B = np.zeros((n, n))
    A = np.eye(n)
    c = np.exp(-r*t)
    B += c * A
    
    for k in range(1, M + 1):
        A = A @ P_hat
        c = c * (r*t) / k
        B += c * A
    
    return B

tiempos = [0.5, 1.0, 5.0]
res_3 = {}

for t in tiempos:
    M = M_propuesto(r, t)
    B = uniformizacion_M(P_hat, r, t, M)
    res_3[t] = {'P': B, 'M': M}
    print(f'P({t}) usando M = {M}')
    print(B)
    print('Suma por renglón =', B.sum(axis=1))
    print()

P(0.5) usando M = 20
[[0.25060868 0.2169646  0.38665694 0.14576979]
 [0.25313484 0.23836098 0.37440924 0.13409493]
 [0.1691195  0.19361489 0.42030102 0.2169646 ]
 [0.15801748 0.15744464 0.39833179 0.28620609]]
Suma por renglón = [1. 1. 1. 1.]

P(1.0) usando M = 20
[[0.20615112 0.20390203 0.3987096  0.1912358 ]
 [0.20828421 0.2053407  0.39789917 0.18847446]
 [0.19675849 0.19837934 0.40095869 0.20390203]
 [0.19204622 0.19399715 0.40147094 0.21248423]]
Suma por renglón = [0.99999854 0.99999854 0.99999854 0.99999854]

P(5.0) usando M = 58
[[0.19999963 0.19999963 0.39999925 0.19999962]
 [0.19999963 0.19999963 0.39999925 0.19999962]
 [0.19999962 0.19999962 0.39999925 0.19999963]
 [0.19999962 0.19999962 0.39999925 0.19999963]]
Suma por renglón = [0.99999812 0.99999812 0.99999812 0.99999812]



Los valores de $M$ usados en el ejercicio 3 son:

In [3]:
tabla_M_3 = pd.DataFrame({
    't': tiempos,
    'rt': [r*t for t in tiempos],
    'M usado': [res_3[t]['M'] for t in tiempos]
})

tabla_M_3

,t,rt,M usado
0,0.5,3.0,20
1,1.0,6.0,20
2,5.0,30.0,58


Verificando numéricamente la ecuación de Chapman-Kolmogorov:

$$
P(1)=P(0.5)P(0.5).
$$

In [4]:
P05_3 = res_3[0.5]['P']
P1_3 = res_3[1.0]['P']

producto_3 = P05_3 @ P05_3
diferencia_3 = P1_3 - producto_3
error_ck_3 = np.max(np.abs(diferencia_3))

print('P(0.5)P(0.5) =')
print(producto_3)
print('\nP(1) - P(0.5)P(0.5) =')
print(diferencia_3)
print('\nError máximo absoluto =', error_ck_3)

P(0.5)P(0.5) =
[[0.20615141 0.20390232 0.39871018 0.19123609]
 [0.20828451 0.20534099 0.39789976 0.18847475]
 [0.19675878 0.19837963 0.40095927 0.20390232]
 [0.19204651 0.19399744 0.40147152 0.21248452]]

P(1) - P(0.5)P(0.5) =
[[-0.00000029 -0.00000029 -0.00000058 -0.00000029]
 [-0.00000029 -0.00000029 -0.00000058 -0.00000029]
 [-0.00000029 -0.00000029 -0.00000058 -0.00000029]
 [-0.00000029 -0.00000029 -0.00000058 -0.00000029]]

Error máximo absoluto = 5.8203336378293e-07


$\textbf{Como el error máximo absoluto es muy pequeño, la igualdad de Chapman-Kolmogorov se verifica numéricamente. La diferencia observada se debe al truncamiento de la serie infinita.}$

## Ejercicio 4.2

Resolvamos el ejercicio 3 aplicando el algoritmo de uniformización con tolerancia

$$
\varepsilon=0.00001.
$$

Debemos elegir $M$ de modo que

$$
\sum_{k=M+1}^{\infty}e^{-rt}\frac{(rt)^k}{k!}\leq \varepsilon.
$$



In [5]:
def uniformizacion_eps(R, t, eps=1e-5):
    tasas = R.sum(axis=1)
    r = tasas.max()
    P_hat = R / r
    np.fill_diagonal(P_hat, 1 - tasas / r)
    
    n = R.shape[0]
    A = P_hat.copy()
    c = np.exp(-r*t)
    B = c * np.eye(n)
    suma = c
    k = 1
    
    while suma < 1 - eps:
        c = c * (r*t) / k
        B = B + c * A
        A = A @ P_hat
        suma = suma + c
        k = k + 1
    
    M = k - 1
    cola = 1 - suma
    return B, M, cola, r, P_hat

eps = 1e-5
res_42 = {}

for t in tiempos:
    B, M, cola, r_calculada, P_hat_calculada = uniformizacion_eps(R, t, eps)
    res_42[t] = {'P': B, 'M': M, 'cola': cola}
    print(f'P({t}) con eps = {eps}, M = {M}')
    print(B)
    print('Suma por renglón =', B.sum(axis=1))
    print('Cola poissoniana =', cola)
    print()

P(0.5) con eps = 1e-05, M = 13
[[0.250608   0.21696392 0.38665557 0.14576911]
 [0.25313416 0.2383603  0.37440788 0.13409425]
 [0.16911882 0.19361421 0.42029966 0.21696392]
 [0.1580168  0.15744396 0.39833043 0.28620541]]
Suma por renglón = [0.9999966 0.9999966 0.9999966 0.9999966]
Cola poissoniana = 3.4019146132324707e-06

P(1.0) con eps = 1e-05, M = 19
[[0.20615038 0.20390128 0.39870811 0.19123506]
 [0.20828347 0.20533995 0.39789768 0.18847371]
 [0.19675775 0.19837859 0.4009572  0.20390128]
 [0.19204548 0.1939964  0.40146945 0.21248349]]
Suma por renglón = [0.99999482 0.99999482 0.99999482 0.99999482]
Cola poissoniana = 5.180168936913532e-06

P(5.0) con eps = 1e-05, M = 56
[[0.19999853 0.19999853 0.39999705 0.19999852]
 [0.19999853 0.19999853 0.39999705 0.19999852]
 [0.19999852 0.19999852 0.39999705 0.19999853]
 [0.19999852 0.19999852 0.39999705 0.19999853]]
Suma por renglón = [0.99999262 0.99999262 0.99999262 0.99999262]
Cola poissoniana = 7.378955002357301e-06



Los valores de $M$ elegidos por la tolerancia son:

In [6]:
tabla_M_42 = pd.DataFrame({
    't': tiempos,
    'rt': [r*t for t in tiempos],
    'epsilon': [eps for _ in tiempos],
    'M elegido': [res_42[t]['M'] for t in tiempos],
    'cola poissoniana': [res_42[t]['cola'] for t in tiempos],
    'cumple cola <= epsilon': [res_42[t]['cola'] <= eps for t in tiempos]
})

tabla_M_42

,t,rt,epsilon,M elegido,cola poissoniana,cumple cola <= epsilon
0,0.5,3.0,0.00001,13,0.000003,True
1,1.0,6.0,0.00001,19,0.000005,True
2,5.0,30.0,0.00001,56,0.000007,True


Verificamos con estas matrices la ecuación de Chapman-Kolmogorov:

$$
P(1)=P(0.5)P(0.5).
$$

In [8]:
P05_42 = res_42[0.5]['P']
P1_42 = res_42[1.0]['P']

producto_42 = P05_42 @ P05_42
diferencia_42 = P1_42 - producto_42
error_ck_42 = np.max(np.abs(diferencia_42))

print('P(0.5)P(0.5), usando el algoritmo con tolerancia =')
print(producto_42)
print('\nP(1) - P(0.5)P(0.5), usando el algoritmo con tolerancia =')
print(diferencia_42)
print('\nError máximo absoluto =', error_ck_42)

P(0.5)P(0.5), usando el algoritmo con tolerancia =
[[0.20615005 0.20390096 0.39870746 0.19123473]
 [0.20828314 0.20533963 0.39789704 0.18847339]
 [0.19675742 0.19837827 0.40095655 0.20390096]
 [0.19204515 0.19399608 0.4014688  0.21248316]]

P(1) - P(0.5)P(0.5), usando el algoritmo con tolerancia =
[[0.00000032 0.00000032 0.00000065 0.00000032]
 [0.00000032 0.00000032 0.00000065 0.00000032]
 [0.00000032 0.00000032 0.00000065 0.00000032]
 [0.00000032 0.00000032 0.00000065 0.00000032]]

Error máximo absoluto = 6.494595712336348e-07


Comparando los resultados del ejercicio 3, obt. 

In [9]:
comparacion = []

for t in tiempos:
    P_3 = res_3[t]['P']
    P_42 = res_42[t]['P']
    dif = P_3 - P_42
    comparacion.append({
        't': t,
        'M ejercicio 3': res_3[t]['M'],
        'M ejercicio 4.2': res_42[t]['M'],
        'cola 4.2': res_42[t]['cola'],
        'max |P_3(t)-P_4.2(t)|': np.max(np.abs(dif))
    })

pd.DataFrame(comparacion)

,t,M ejercicio 3,M ejercicio 4.2,cola 4.2,max |P_3(t)-P_4.2(t)|
0,0.5,20,13,0.000003,0.000001
1,1.0,20,19,0.000005,0.000001
2,5.0,58,56,0.000007,0.000002


In [10]:
for t in tiempos:
    print(f'Diferencia P_3({t}) - P_4.2({t}) =')
    print(res_3[t]['P'] - res_42[t]['P'])
    print()

Diferencia P_3(0.5) - P_4.2(0.5) =
[[0.00000068 0.00000068 0.00000136 0.00000068]
 [0.00000068 0.00000068 0.00000136 0.00000068]
 [0.00000068 0.00000068 0.00000136 0.00000068]
 [0.00000068 0.00000068 0.00000136 0.00000068]]

Diferencia P_3(1.0) - P_4.2(1.0) =
[[0.00000075 0.00000075 0.00000149 0.00000075]
 [0.00000075 0.00000075 0.00000149 0.00000075]
 [0.00000075 0.00000075 0.00000149 0.00000075]
 [0.00000075 0.00000075 0.00000149 0.00000075]]

Diferencia P_3(5.0) - P_4.2(5.0) =
[[0.0000011 0.0000011 0.0000022 0.0000011]
 [0.0000011 0.0000011 0.0000022 0.0000011]
 [0.0000011 0.0000011 0.0000022 0.0000011]
 [0.0000011 0.0000011 0.0000022 0.0000011]]



$\textbf{Los valores obtenidos por ambos procedimientos son prácticamente iguales. }$

$\textbf{La ecuación de Chapman-Kolmogorov también se verifica numéricamente con el algoritmo de tolerancia; las diferencias restantes corresponden al truncamiento de las series.}$